In [10]:
import pandas as pd
from pathlib import Path

# 1. Get the directory where THIS script is saved
data_dir = Path.cwd().parent / 'Data'

# 3. Load the files
dataframes = {}
try:
    for file_path in data_dir.glob('*.csv'):
        # Use the filename without the (.csv) as the dictionary key
        file_name = file_path.stem
        dataframes[file_name] = pd.read_csv(file_path, encoding='latin1')  #Encoding to ensure the file will be loaded
        print(f'Successfully loaded the file: {file_name}')

        # Access the data easily.
        # roster_df = dataframes['Roster']
except Exception as e:
    print(f'An error occurred loading the file: {e}')

Successfully loaded the file: Contacts
Successfully loaded the file: Productivity
Successfully loaded the file: Roster


In [11]:
# Checking all data at once
for name, df in dataframes.items():
    print(f"--- {name} ---")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    print(df.columns.tolist())
    print("\n")

--- Contacts ---
Rows: 248397, Columns: 12
['Agent', 'Date', 'LOB', 'Channel', 'ACW', 'Dual/Multiple_Chat_AHT', 'Inbound_Transaction', 'Outbound_Transaction', 'Handle_Time', 'Hold_Time', 'Outbound_Handle_Time', 'Missed_Contacts']


--- Productivity ---
Rows: 183860, Columns: 16
['Agent', 'Date', 'Aux_Duration', '1st_BreakDuration', '2nd_Break_Duration', '3rd_Break_Duration', 'Email_Duration', 'Lunch_Duration', 'Meeting_Duration_', 'Technical_Issue_Duration', 'Personal_Duration', 'Task_Duration', 'Training_Duration', 'Available_Duration', 'Busy_Duration', 'Login_Duration']


--- Roster ---
Rows: 2544, Columns: 12
['Agent', 'Team Manager', 'Active Date', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Days', 'Tenurity']




In [12]:
# Create a copy of the file to start data cleaning
staging_df = {}
for name, df in dataframes.items():
    staging_df[name + '1'] = df.copy()  

In [14]:
# 1. Clean Columns Names (remove space/special characters) and update to lowercase
for name, df in staging_df.items():
    df.columns = df.columns.str.lower().str.strip().str.replace(r'[^\w]+', '_', regex=True).str.strip('_')

In [17]:
# 2. Remove Columns and Rows with NULL values ONLY
for name in staging_df:
    # Remove Columns
    staging_df[name] = staging_df[name].dropna(how='all', axis=1)
    # Remove rows
    staging_df[name] = staging_df[name].dropna(how='all', axis=0)

In [20]:
# 3. Drop Unnecessary column in Roster.csv
staging_df['Roster1'] = staging_df['Roster1'].drop(columns=['days', 'tenurity'], errors='ignore')

In [25]:
# 4. Update column `date` to datatype Datetime
col_date = ['date', 'active_date']

for name, df in staging_df.items():
    for col in col_date:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f'Converted {col} to {name}')

for name in staging_df:
    print()
    print(staging_df[name].info())

Converted date to Contacts1
Converted date to Productivity1
Converted active_date to Roster1

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248397 entries, 0 to 248396
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   agent                   248397 non-null  object        
 1   date                    248397 non-null  datetime64[ns]
 2   lob                     248397 non-null  object        
 3   channel                 248397 non-null  object        
 4   acw                     248397 non-null  int64         
 5   dual_multiple_chat_aht  248397 non-null  int64         
 6   inbound_transaction     248397 non-null  int64         
 7   outbound_transaction    248397 non-null  int64         
 8   handle_time             248397 non-null  int64         
 9   hold_time               248397 non-null  int64         
 10  outbound_handle_time    248397 non-null  int64         
 1

In [31]:
# CLEAN & AGGREGATE duplicate entry in Productivity File
prod_clean = (staging_df['Productivity1'].groupby(['agent', 'date']).sum(numeric_only=True).reset_index())

In [32]:
# Merge Contacts1 and Productivity1 to create a Master file then merge with Roster1
master_file = staging_df['Contacts1'].merge(prod_clean, on=['agent', 'date'], how='left')
master_file = master_file.merge(staging_df['Roster1'], on='agent', how='left')

print(len(master_file))
print()
print(len(staging_df['Contacts1']))
print()
print(len(staging_df['Productivity1']))

248397

248397

183860


In [39]:
print(master_file.columns.tolist())

['agent', 'date', 'lob', 'channel', 'acw', 'dual_multiple_chat_aht', 'inbound_transaction', 'outbound_transaction', 'handle_time', 'hold_time', 'outbound_handle_time', 'missed_contacts', 'aux_duration', '1st_breakduration', '2nd_break_duration', '3rd_break_duration', 'email_duration', 'lunch_duration', 'meeting_duration', 'technical_issue_duration', 'personal_duration', 'task_duration', 'training_duration', 'available_duration', 'busy_duration', 'login_duration', 'team_manager', 'active_date']


In [43]:
# Save the clean master_file for PowerBI dashboard
master_file.to_csv(data_dir / 'Master File.csv', index=False)